In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive', timeout_ms=240000)
auth.authenticate_user()
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/PHD/CSP3_GPP_outputs')
OUTPUT=ROOT/'CH3_profitability/20261007_profitability_v1'
print('Drive restored; existing release preserved',flush=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive restored; existing release preserved


In [ ]:
import sys, subprocess, json, hashlib, zipfile, importlib, importlib.metadata as metadata
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(OUTPUT/'code/requirements.txt')])
package_sha='09a3824244b04f1b4e4c1eed0113e3727e72b0e9932e327850df4f9a2989a831'
archive=OUTPUT/('publication_package_'+package_sha[:16]+'.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest()==package_sha
CODE=Path('/content/ch3_publication_'+package_sha[:16])
with zipfile.ZipFile(archive) as z: z.extractall(CODE)
manifest=json.loads((CODE/'publication_manifest.json').read_text())
for row in manifest:
    assert hashlib.sha256((CODE/row['file']).read_bytes()).hexdigest()==row['sha256'],row['file']
sys.path.insert(0,str(CODE)); importlib.invalidate_caches()
import ee
ee.Initialize(project='ee-njberkowitz95')
print('Earth Engine ready; immutable publication code verified',flush=True)
print('Recovery record exists:',(OUTPUT/'verification/cloud_write_recovery.json').exists(),flush=True)
print('GeoPackage bytes:',(OUTPUT/'CH3_spatial.gpkg').stat().st_size,flush=True)


Earth Engine ready; immutable publication code verified
Recovery record exists: True
GeoPackage bytes: 3808022528


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
import json, hashlib, zipfile, sys, importlib
package_sha='e1a530140a777a9547e817738d11edaa36a08d3058004cafc09a0b1579b08c1c'
archive=OUTPUT/('publication_package_'+package_sha[:16]+'.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest()==package_sha
CODE=Path('/content/ch3_publication_'+package_sha[:16])
with zipfile.ZipFile(archive) as z: z.extractall(CODE)
manifest=json.loads((CODE/'publication_manifest.json').read_text())
for row in manifest:
    assert hashlib.sha256((CODE/row['file']).read_bytes()).hexdigest()==row['sha256'],row['file']
for name in list(sys.modules):
    if name.startswith(('ch3_profitability','ch4_marginality')): del sys.modules[name]
sys.path.insert(0,str(CODE)); importlib.invalidate_caches()
from ch4_marginality.pipeline import sha, dump
recovery=json.loads((OUTPUT/'verification/cloud_write_recovery.json').read_text())
audit_record=json.loads((OUTPUT/'verification/release_audit.json').read_text())
assert recovery['verified'] and recovery['closed_sqlite_connection']
assert audit_record['verified'] and audit_record['geopackage_table_counts']=={'annual_profitability':58,'county_sensitivity':6786,'patch_sensitivity':5347656}
assert sha(OUTPUT/'CH3_spatial.gpkg')==recovery['geopackage_sha256']==audit_record['geopackage_sha256']
assert sha(OUTPUT/'tables/patch_sensitivity.csv.gz')==audit_record['patch_table_sha256']
dump(OUTPUT/'publication_package_record.json',{'archive':archive.name,'sha256':package_sha,'files':len(manifest),'verified_release_audit_reused':True})
print('Final closed outputs match the independent release audit',flush=True)
print(audit_record,flush=True)


Final closed outputs match the independent release audit
{'verified': True, 'economic_combinations': 522, 'baseline_combinations': 58, 'spatial_levels_reconciled': True, 'geopackage_table_counts': {'annual_profitability': 58, 'county_sensitivity': 6786, 'patch_sensitivity': 5347656}, 'geopackage_sha256': '86ebacc101c92753330bd096e6e88979a66206900f254d6f861b935589e5c951', 'patch_table_sha256': '20e88a0a01892c926ee7e31a82bfe30c504dcf18da9ee480b21581ecf379b7d1', 'histogram_tails_reconciled': True, 'missing_cash_retained': True, 'temporal_gaps_and_experimental_exclusion_verified': True, 'executed_code_manifest_sha256': 'd4588c5e8947ee06af3ce7d17f5090953e860a74017c423f03ff9d64f484e0be', 'method_provenance': 'Frozen code/ package and code_manifest.json; publication helpers are separately versioned'}


In [7]:
import time, json, importlib.metadata as metadata
import numpy as np, pandas as pd
from ch4_marginality.pipeline import sha, dump
from ch4_marginality.profit_workspace import lock_inputs
from ch3_profitability.pipeline import SOURCE_RELEASE
from ch3_profitability.assets import verify
from ch3_profitability.verify_native import run as verify_native
from ch3_profitability.build_app import build
audit_record=json.loads((OUTPUT/'verification/release_audit.json').read_text())
assert audit_record['verified'] and audit_record.get('geopackage_table_counts')
assert json.loads((OUTPUT/'verification/cloud_write_recovery.json').read_text())['closed_sqlite_connection']
stage=json.loads((OUTPUT/'verification/app_staging.json').read_text())
assert stage['verified']
for row in stage['files']:
    assert sha(OUTPUT/'app_assets'/row['file'])==row['sha256'],row['file']
versions={name:metadata.version(name) for name in ['earthengine-api','google-cloud-storage','numpy','pandas','rasterio','geopandas']}
assert np.__version__==versions['numpy'] and pd.__version__==versions['pandas']
dump(OUTPUT/'publication_environment.json',versions)
prior=ROOT/'CH4_marginality'/SOURCE_RELEASE
lock_inputs(ROOT,prior/'analysis',OUTPUT,json.loads((prior/'input_manifest.json').read_text()))
tasks=json.loads((OUTPUT/'asset_tasks.json').read_text())
print('Resuming existing import task IDs; no duplicate submission',flush=True)
for attempt in range(360):
    ids=[x['response']['id'] for x in tasks if 'response' in x]
    states=ee.data.getTaskStatus(ids) if ids else []
    if attempt%6==0 or all(x['state']=='COMPLETED' for x in states):
        print([(x['state'],x.get('error_message')) for x in states],flush=True)
    if any(x['state'] in ['FAILED','CANCELLED'] for x in states):
        raise RuntimeError('An Earth Engine import failed')
    if all(x['state']=='COMPLETED' for x in states): break
    time.sleep(10)
else: raise RuntimeError('Imports remain pending; resume the existing IDs')
dump(OUTPUT/'asset_task_completion.json',{'verified':True,'tasks':states})
print(verify(OUTPUT),flush=True)
verify_native(ROOT)
lock_inputs(ROOT,prior/'analysis',OUTPUT,json.loads((prior/'input_manifest.json').read_text()))
print('Verified candidate:',build(OUTPUT),flush=True)
print('Browser candidate testing and live app publication remain required',flush=True)

Resuming existing import task IDs; no duplicate submission
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETED', None), ('COMPLETED', None), ('RUNNING', None)]
[('COMPLETE

In [8]:
import sys, json, hashlib, zipfile, importlib, importlib.metadata as metadata
package_sha='e10b34301b947b297ff0f1336f1fceec5c7d815ddff75a09ca0f25992d69755a'
archive=OUTPUT/('publication_package_'+package_sha[:16]+'.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest()==package_sha
CODE=Path('/content/ch3_publication_'+package_sha[:16])
with zipfile.ZipFile(archive) as z: z.extractall(CODE)
manifest=json.loads((CODE/'publication_manifest.json').read_text())
for row in manifest:
    assert hashlib.sha256((CODE/row['file']).read_bytes()).hexdigest()==row['sha256'],row['file']
for line in (CODE/'requirements.txt').read_text().splitlines():
    if '==' in line:
        name,version=line.split('=='); assert metadata.version(name)==version
for name in list(sys.modules):
    if name.startswith(('ch3_profitability','ch4_marginality')): del sys.modules[name]
sys.path.insert(0,str(CODE)); importlib.invalidate_caches()
from ch4_marginality.pipeline import dump, sha
from ch4_marginality.profit_workspace import lock_inputs
from ch3_profitability.pipeline import SOURCE_RELEASE
from ch3_profitability.build_app import build
assert json.loads((OUTPUT/'asset_verification.json').read_text())['verified']
prior=ROOT/'CH4_marginality'/SOURCE_RELEASE
lock_inputs(ROOT,prior/'analysis',OUTPUT,json.loads((prior/'input_manifest.json').read_text()))
candidate=build(OUTPUT)
dump(OUTPUT/'publication_package_record.json',dict(archive=archive.name,sha256=package_sha,files=len(manifest),prior_asset_verification_archive='publication_package_e1a530140a777a95.zip',candidate_sha256=sha(candidate)))
print('Final candidate with current tested interface:',candidate,flush=True)
print('Candidate SHA256:',sha(candidate),flush=True)
print('Live app testing and deployment remain required',flush=True)

Final candidate with current tested interface: /content/ch3_publication_e10b34301b947b29/gee/gee_app_Yield_PEM.js
Candidate SHA256: 654c4444c85d4e516f5726132e4b863e435dd627f13c1e6eb220897092387c95
Live app testing and deployment remain required
